# Entraînement YOLOv8 — Passages piétons (DefisAccess)

**Avant de commencer :** Dans le menu Colab → *Exécution* → *Modifier le type d'exécution* → choisir **GPU (T4)**

Ensuite exécutez les cellules dans l'ordre (Shift+Entrée).

## Étape 1 — Vérifier le GPU

In [ ]:
!nvidia-smi
# Vous devez voir une carte GPU (ex: Tesla T4). Si erreur, activez le GPU dans les paramètres.

## Étape 2 — Installer Ultralytics (YOLOv8)

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()  # vérifie que CUDA est bien détecté

## Étape 3 — Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Vérifier que les fichiers sont bien présents
import os
base = '/content/drive/MyDrive/DefiAccess2/testeIAPP'

print('data.yaml :', os.path.exists(f'{base}/data.yaml'))
print('Images train :', len(os.listdir(f'{base}/yolo/images/train')), 'fichiers')
print('Images val   :', len(os.listdir(f'{base}/yolo/images/val')), 'fichiers')
print('Labels train :', len(os.listdir(f'{base}/yolo/labels/train')), 'fichiers')

## Étape 4 — Entraînement

YOLOv8-nano (`yolov8n.pt`) est le plus rapide. Durée estimée : **10-20 minutes** avec GPU T4.

In [ ]:
from ultralytics import YOLO

base = '/content/drive/MyDrive/DefiAccess2/testeIAPP'

model = YOLO('yolov8n.pt')  # télécharge automatiquement le modèle de base

results = model.train(
    data=f'{base}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='passage_pieton_v2',
    patience=20,       # arrêt anticipé si pas d'amélioration après 20 epochs
    save=True,
    plots=True,
)

print('\nEntraînement terminé !')
print('Meilleur modèle :', results.save_dir)

## Étape 5 — Évaluer les résultats

Regardez les métriques clés :
- **mAP50 > 0.5** = bon modèle
- **mAP50 > 0.7** = très bon modèle

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob

# Charger le meilleur modèle sauvegardé
best_path = sorted(glob.glob('runs/detect/passage_pieton_v2*/weights/best.pt'))[-1]
print('Modèle chargé :', best_path)

model_eval = YOLO(best_path)
metrics = model_eval.val()

print(f"\nmAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")
print(f"Précision  : {metrics.box.mp:.3f}")
print(f"Rappel     : {metrics.box.mr:.3f}")

In [ ]:
# Afficher les courbes d'entraînement
import glob
from IPython.display import Image, display

run_dir = sorted(glob.glob('runs/detect/passage_pieton_v2*'))[-1]

for plot in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    path = f'{run_dir}/{plot}'
    if os.path.exists(path):
        print(f'--- {plot} ---')
        display(Image(path, width=800))

## Étape 6 — Sauvegarder `best.pt` sur Google Drive ET le télécharger

In [ ]:
import shutil, glob

best_path = sorted(glob.glob('runs/detect/passage_pieton_v2*/weights/best.pt'))[-1]

# Copier sur Google Drive (sauvegarde)
drive_dest = '/content/drive/MyDrive/DefiAccess2/testeIAPP/best.pt'
shutil.copy(best_path, drive_dest)
print('Sauvegardé sur Drive :', drive_dest)

# Télécharger directement dans le navigateur
from google.colab import files
files.download(best_path)
print('Téléchargement lancé !')

## Étape suivante (en local)

Une fois `best.pt` téléchargé, remplacez le fichier dans votre projet local :
```
Prototype_Appli_DefisAccess/models/best.pt
```
Le script `src/IA_PP.py` chargera automatiquement ce nouveau modèle.